# FTSE 100 · ESG · Machine Learning — Dissertation Analysis Notebook

**Navya Shahi — MSc International Corporate Finance and Banking, University of Glasgow**

This notebook contains the **exact code that produced the numbers, tables and figures in the dissertation** (Chapters 3–5). Every code cell is the original script text, unmodified, except where a cell is explicitly marked **(path change only)** or **(added)**.

## How to use
1. Unzip the package so the folder looks like this, then open this notebook **from inside that folder**:
```
FTSE100_notebook/
├── FTSE100_ESG_ML_Pipeline.ipynb   <- this file
├── data/Data.xlsx                  <- your raw LSEG Workspace export (3 sheets)
├── saved_colab_outputs/            <- your real XGBoost + SHAP files from Google Colab
└── expected_outputs/               <- results from the original run, used for the self-check at the end
```
2. Run the cells **top to bottom** (Kernel → Restart & Run All also works).
3. The last cell compares everything this notebook produced against `expected_outputs/` and prints the maximum difference per file.

## What each step produces (and where it appears in the dissertation)
| Step | Produces | Dissertation |
|---|---|---|
| 1 / 1b | parsed ESG, financial and sector data | Ch. 3.1–3.2 |
| 2 | merged panel + forward-return target (rank-based merge) | Ch. 3.2 |
| 3 | cleaned modelling panel (875 obs, 15 ESG fields, sector dummies) | Ch. 3.4 |
| 4 | OLS / LASSO / Random Forest / gradient boosting, walk-forward CV | Tables 5.1, 5.2 |
| 5 | portfolio backtest | Table 5.8 |
| 6 | pre/post-2020 robustness | Table 5.6 |
| 7 | significance tests (sklearn stand-in version) | superseded by Step 10 |
| 8 | descriptive statistics, correlations | Tables 3.2, 3.4 |
| 9 | **real** XGBoost + SHAP (needs `xgboost`, `shap`) | Tables 5.1, 5.2, 5.4 |
| 10 | merge, significance tests and backtest with real XGBoost | Tables 5.3, 5.7, 5.8 |
| 11 | recap tables + self-check | — |

## Please read — two honest notes
* **Environment.** The numbers were verified to reproduce exactly (maximum difference 0.0 on every output file) with Python 3 · pandas 3.0.2 · NumPy 2.4.4 · scikit-learn 1.8.0 · SciPy 1.17.1. Other library versions can change results in the 3rd–4th decimal place (tree models and LASSO are sensitive to version).
* **Real XGBoost + SHAP (Step 9).** The XGBoost figures in the dissertation (mean OOS R² = 0.036 and the SHAP rankings) came from a Google Colab run. If you re-run Step 9 on your machine with a different `xgboost` version the figures may differ slightly. **Step 9b** restores your original Colab outputs from `saved_colab_outputs/`, so Steps 10–11 reproduce the dissertation numbers exactly either way.

## 0 · Setup (run once)

In [ ]:
# Install anything missing (safe to re-run). xgboost + shap are only needed for Step 9a.
%pip install pandas numpy scipy scikit-learn openpyxl matplotlib xgboost shap --quiet

In [ ]:
import os
for d in ["data", "outputs"]:
    os.makedirs(d, exist_ok=True)

assert os.path.exists("data/Data.xlsx"), "Put your Data.xlsx (the 3-sheet LSEG export) in the data/ folder next to this notebook."
print("Working directory:", os.getcwd())
print("data/Data.xlsx found:", os.path.getsize("data/Data.xlsx"), "bytes")

## Step 1 · Parse the raw LSEG export (original script `01_parse_raw_export.py`)
Reads the ESG sheet and the financial sheet out of the Excel formula block. *Note: this original script does not read the sector sheet — that was done by Step 1b, which is the version used for the final run.*

In [ ]:
"""
01_parse_raw_export.py

Reads the raw Excel export from LSEG Workspace (Data.xlsx, with an ESG
sheet and a financial-data sheet) and parses it into two clean pandas
DataFrames: esg_raw.pkl and fin_raw.pkl.

WHY THIS IS NEEDED: the @RDP.Data formula in Excel outputs a somewhat
messy block (a leftover stale RIC column in column A left over from
the original paste, a header row that isn't row 1, field names as
column headers starting partway across the sheet). This script finds
the real header row and data block automatically and reshapes it into
a standard long-format panel: one row per (company, fiscal-year-end
date), one column per field.

INPUT:  data/Data.xlsx  -- must have your ESG pull as the first sheet
        and your financial pull as another sheet (edit SHEET_ESG /
        SHEET_FIN below if your sheet names differ).
OUTPUT: esg_raw.pkl, fin_raw.pkl (pickled DataFrames, easy for the
        next script to load without re-parsing Excel every time)
"""

import openpyxl
import pandas as pd

SHEET_ESG = "Sheet1"
SHEET_FIN = "Sheet2"
N_ESG_FIELDS = 20   # adjust if you add/remove ESG fields in your pull
N_FIN_FIELDS = 3    # Price Close, Company Market Cap, Total Return

def parse_rdp_sheet(ws, n_fields):
    """
    RDP.Data output structure in this workbook:
      - some early rows are leftover stale RIC list (column A only, ignore)
      - one row has 'Updated at ...' in column C -- the row right after
        that is the actual field-name header, in columns E onward
      - data rows have: col C = RIC, col D = date, cols E.. = field values
    """
    header = None
    rows = []
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row, values_only=True):
        marker = row[2]
        if isinstance(marker, str) and marker.startswith("Updated at"):
            header = row[4:4 + n_fields]
            continue
        ric, date = row[2], row[3]
        if ric and date and hasattr(date, "year"):
            rows.append((ric, date) + row[4:4 + n_fields])
    cols = ["ric", "date"] + list(header)
    return pd.DataFrame(rows, columns=cols)

def main():
    wb = openpyxl.load_workbook("data/Data.xlsx", data_only=True)

    esg_df = parse_rdp_sheet(wb[SHEET_ESG], N_ESG_FIELDS)
    print(f"ESG data: {esg_df.shape[0]} rows, {esg_df['ric'].nunique()} companies")

    fin_df = parse_rdp_sheet(wb[SHEET_FIN], N_FIN_FIELDS)
    print(f"Financial data: {fin_df.shape[0]} rows, {fin_df['ric'].nunique()} companies")

    esg_df.to_pickle("data/esg_raw.pkl")
    fin_df.to_pickle("data/fin_raw.pkl")
    print("Saved: data/esg_raw.pkl, data/fin_raw.pkl")

if __name__ == "__main__":
    main()

## Step 1b · Parse all three sheets including sector (the version used for the final run)
This is the exact inline code that was run to parse the ESG sheet (`Sheet1`), financial sheet (`Sheet2`) **and** the GICS sector sheet (`Sheet3`). It overwrites the Step 1 pickles with identical ESG/financial data and adds `data/sector_raw.pkl`, which Step 2 requires.

In [ ]:
import openpyxl
import pandas as pd

wb = openpyxl.load_workbook('data/Data.xlsx', data_only=True)

def parse_rdp_sheet(ws, n_fields):
    header = None
    rows = []
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row, values_only=True):
        marker = row[2]
        if isinstance(marker, str) and marker.startswith("Updated at"):
            header = row[4:4+n_fields]
            continue
        ric, date = row[2], row[3]
        if ric and date and hasattr(date, 'year'):
            rows.append((ric, date) + row[4:4+n_fields])
    cols = ['ric','date'] + list(header)
    return pd.DataFrame(rows, columns=cols)

def parse_sector_sheet(ws):
    rows = []
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row, values_only=True):
        ric, sector = row[2], row[3]
        if ric and sector and not (isinstance(ric,str) and ric.startswith('Updated at')):
            rows.append((ric, sector))
    return pd.DataFrame(rows, columns=['ric','sector'])

esg_df = parse_rdp_sheet(wb['Sheet1'], 20)
fin_df = parse_rdp_sheet(wb['Sheet2'], 3)
sector_df = parse_sector_sheet(wb['Sheet3'])
sector_df['sector'] = sector_df['sector'].replace('NULL', pd.NA)

print("ESG:", esg_df.shape, esg_df['ric'].nunique(), "companies")
print("Financial:", fin_df.shape, fin_df['ric'].nunique(), "companies")
print("Sector:", sector_df.shape, sector_df['ric'].nunique(), "companies,", sector_df['sector'].isna().sum(), "missing")

esg_df.to_pickle('data/esg_raw.pkl')
fin_df.to_pickle('data/fin_raw.pkl')
sector_df.to_pickle('data/sector_raw.pkl')
print("Saved all three.")

## Step 2 · Merge ESG + financial + sector, build the forward-return target (original `02_merge_and_build_target.py`)
Rank-based merge (ESG data uses fiscal year-end, market data uses calendar year-end), forward return with no look-ahead. Expected: **879 rows, 99 companies, fiscal years 2015–2025** (Table 3.1).

In [ ]:
"""
02_merge_and_build_target.py

Merges the ESG panel and financial panel, and constructs the forward
excess-return target variable.

KEY DESIGN DECISION -- merge by RANK, not by exact date:
Refinitiv reports ESG/fundamental data on each company's own fiscal
year-end (e.g. III.L reports ESG data as of 31 March), but market data
fields (price, market cap, return) pulled at the same "FY" frequency
snap to calendar year-end instead (e.g. 31 December). These dates
don't match for the same company/year, so merging on exact date would
silently drop most of your data or misalign it.

Instead: within each company, sort both datasets by date descending
(most recent first) and assign a period_rank (0 = most recent year,
1 = one year back, etc). Since both pulls used the identical
SDate=0,EDate=-9,Frq=FY parameters for the same companies, rank
position corresponds to the same relative fiscal-year offset in both
datasets even though the calendar dates differ. Merge on (ric, rank).

FORWARD RETURN CONSTRUCTION (no look-ahead):
rank 0 = most recent year (e.g. FY2025), rank 1 = the year before that
(FY2024), etc. To predict "forward" return using CURRENT-year ESG
data, the target for the row at rank r is the Total Return realised in
the NEXT, more recent period -- i.e. rank r-1. This means rank 0 (the
most recent year) has no forward return yet (we don't know next
year's outcome) and gets dropped -- which is correct: you cannot
predict a return you don't have.

INPUT:  data/esg_raw.pkl, data/fin_raw.pkl (from script 01)
OUTPUT: data/analysis_panel.csv -- one row per (company, fiscal year)
        with ESG features + the fwd_total_return target
"""

import pandas as pd
import numpy as np

TARGET = "fwd_total_return"

def main():
    esg_df = pd.read_pickle("data/esg_raw.pkl")
    fin_df = pd.read_pickle("data/fin_raw.pkl")
    sector_df = pd.read_pickle("data/sector_raw.pkl")  # one row per company, no date/rank needed

    esg_df = esg_df.sort_values(["ric", "date"], ascending=[True, False])
    esg_df["period_rank"] = esg_df.groupby("ric").cumcount()

    fin_df = fin_df.sort_values(["ric", "date"], ascending=[True, False])
    fin_df["period_rank"] = fin_df.groupby("ric").cumcount()

    merged = esg_df.merge(fin_df, on=["ric", "period_rank"], suffixes=("_esg", "_fin"))
    merged = merged.sort_values(["ric", "period_rank"])

    # forward return: shift(1) within each company grabs the value from
    # the row one rank MORE RECENT (since rank ascends with age)
    merged[TARGET] = merged.groupby("ric")["Total Return"].shift(1)

    panel = merged[merged[TARGET].notna()].copy()
    panel["fiscal_year"] = panel["date_esg"].dt.year

    # attach sector (static per company, no rank/date matching needed)
    panel = panel.merge(sector_df, on="ric", how="left")
    print(f"\nSector coverage: {panel['sector'].notna().sum()}/{len(panel)} rows have a sector "
          f"({panel['sector'].isna().sum()} missing, e.g. trusts/funds without GICS classification)")

    print(f"Final analysis panel: {panel.shape[0]} rows, {panel['ric'].nunique()} companies")
    print(f"Fiscal years: {sorted(panel['fiscal_year'].unique())}")
    print("\nRows per fiscal year:")
    print(panel["fiscal_year"].value_counts().sort_index())

    panel.to_csv("data/analysis_panel.csv", index=False)
    print("\nSaved: data/analysis_panel.csv")

if __name__ == "__main__":
    main()

## Step 3 · Clean the panel (original `03_clean_panel.py`)
Drops the 5 ESG fields with >45% missing, sector-year median imputation, sector dummies, winsorising at 1%/99%. Expected: **875 rows, 15 ESG fields retained**.

In [ ]:
"""
03_clean_panel.py

Takes the merged panel (script 02's output) and prepares it for
modelling: converts boolean policy flags to 0/1, drops ESG fields
missing beyond a threshold, imputes the rest, builds the size control,
and winsorises outliers.

INPUT:  data/analysis_panel.csv
OUTPUT: data/modeling_panel.csv
"""

import pandas as pd
import numpy as np

MISSING_THRESHOLD = 0.45
TARGET = "fwd_total_return"

ESG_COLS_ALL = [
    "Environmental Pillar Score", "Social Pillar Score", "Governance Pillar Score", "ESG Score",
    "CO2 Equivalent Emissions Total", "Independent Board Members", "Board Size", "Renewable Energy Use",
    "Waste Total", "Water Pollutant Emissions", "Health & Safety Policy", "Training Hours Total",
    "Human Rights Policy Score", "Biodiversity Supply Chain Policy", "Product Responsibility Score",
    "Lost Time Injury Rate Total", "Employee Satisfaction", "CSR Sustainability Committee",
    "Shareholder Rights Policy", "Policy Data Privacy Score",
]

def winsorize(s, pct=0.01):
    lo, hi = s.quantile(pct), s.quantile(1 - pct)
    return s.clip(lo, hi)

def main():
    panel = pd.read_csv("data/analysis_panel.csv")
    panel["date_esg"] = pd.to_datetime(panel["date_esg"])

    # boolean policy flags -> 0/1
    for c in ESG_COLS_ALL:
        if panel[c].dtype == object:
            panel[c] = panel[c].map({True: 1, False: 0, "True": 1, "False": 0}).astype(float)

    # drop fields missing beyond threshold
    missing_rate = panel[ESG_COLS_ALL].isna().mean()
    dropped = missing_rate[missing_rate > MISSING_THRESHOLD].index.tolist()
    kept_esg = [c for c in ESG_COLS_ALL if c not in dropped]
    print(f"Dropped {len(dropped)} ESG fields (>{int(MISSING_THRESHOLD*100)}% missing): {dropped}")
    print(f"Retained {len(kept_esg)} ESG fields")

    # sector-year median imputation (falls back to year-median, then
    # global median, for any sector-year cell with too few observations
    # to compute its own median -- common with 12 sectors x 10 years
    # spread across only ~100 firms)
    panel["sector"] = panel["sector"].fillna("Unknown")
    for c in kept_esg:
        panel[c] = panel.groupby(["sector", "fiscal_year"])[c].transform(lambda s: s.fillna(s.median()))
        panel[c] = panel.groupby("fiscal_year")[c].transform(lambda s: s.fillna(s.median()))
        panel[c] = panel[c].fillna(panel[c].median())

    # size control
    panel["log_mktcap"] = np.log(panel["Company Market Cap"].clip(lower=1))

    # sector dummies for the OLS baseline (drop_first avoids the dummy
    # trap; "Unknown" folds into whichever category is dropped)
    sector_dummies = pd.get_dummies(panel["sector"], prefix="sector", drop_first=True)
    panel = pd.concat([panel, sector_dummies], axis=1)
    sector_dummy_cols = list(sector_dummies.columns)

    # winsorize
    for c in kept_esg + ["log_mktcap", TARGET]:
        panel[c] = winsorize(panel[c])

    panel = panel.dropna(subset=[TARGET]).reset_index(drop=True)

    print(f"\nFinal modelling panel: {panel.shape}")
    print(f"Sectors represented: {panel['sector'].value_counts().to_dict()}")
    panel.to_csv("data/modeling_panel.csv", index=False)

    with open("data/sector_dummy_cols.txt", "w") as f:
        f.write("\n".join(sector_dummy_cols))

    with open("data/retained_esg_features.txt", "w") as f:
        f.write("\n".join(kept_esg))
    print("Saved: data/modeling_panel.csv, data/retained_esg_features.txt")

if __name__ == "__main__":
    main()

## Step 4 · Models and walk-forward cross-validation (original `04_run_models.py`)
OLS (year FE + sector dummies), LASSO, Random Forest and the scikit-learn gradient-boosting stand-in, test years 2019–2024. Expected mean OOS R²: OLS −0.071 · LASSO −0.002 · Random Forest 0.019 · GBT stand-in 0.012.

In [ ]:
"""
04_run_models.py

Runs the OLS baseline, LASSO, Random Forest, and gradient boosting
(XGBoost stand-in) on the cleaned panel, using walk-forward
cross-validation: train on all years before the test year, test on
that single year. Same folds used for every model.

NOTE ON XGBOOST/SHAP: this script uses sklearn's
HistGradientBoostingRegressor as a stand-in for xgboost.XGBRegressor,
and sklearn's permutation_importance as a stand-in for SHAP, because
those packages required internet access to install that wasn't
available when this was built. If you're running this on your own
machine with internet access, swap in:
    from xgboost import XGBRegressor
    import shap
for a closer match to your proposal's stated methodology. The
walk-forward evaluation logic is identical either way.

INPUT:  data/modeling_panel.csv, data/retained_esg_features.txt
OUTPUT: outputs/model_results.csv, outputs/oos_predictions.csv,
        outputs/perm_importance_*.csv
"""

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error

TARGET = "fwd_total_return"
TEST_YEAR_MIN = 2019   # first test year -- adjust based on how much
TEST_YEAR_MAX = 2024   # training history you want before the first fold

def oos_r2(y_true, y_pred, y_train_mean):
    naive = np.full_like(y_true, y_train_mean, dtype=float)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - naive) ** 2)
    return 1 - ss_res / ss_tot

def main():
    panel = pd.read_csv("data/modeling_panel.csv")
    with open("data/retained_esg_features.txt") as f:
        kept_esg = [l.strip() for l in f if l.strip()]
    with open("data/sector_dummy_cols.txt") as f:
        sector_dummy_cols = [l.strip() for l in f if l.strip()]
    feature_cols = kept_esg + ["log_mktcap"] + sector_dummy_cols

    all_years = sorted(panel["fiscal_year"].unique())
    test_years = [y for y in all_years if TEST_YEAR_MIN <= y <= TEST_YEAR_MAX]
    print("Test years:", test_years)

    results = []
    predictions_store = []
    last_fold_importance = {}

    for test_year in test_years:
        train = panel[panel["fiscal_year"] < test_year]
        test = panel[panel["fiscal_year"] == test_year]
        if len(train) < 30 or len(test) < 5:
            continue

        X_train_raw = train[feature_cols].values.astype(float)
        X_test_raw = test[feature_cols].values.astype(float)
        y_train = train[TARGET].values.astype(float)
        y_test = test[TARGET].values.astype(float)
        train_mean = y_train.mean()

        # --- OLS with year fixed effects (no firm FE -- too sparse
        # per firm at this sample size; add firm dummies if your real
        # sample grows large enough to support them) ---
        yr_dum_train = pd.get_dummies(train["fiscal_year"], prefix="yr", drop_first=True)
        yr_dum_test = pd.DataFrame(0, index=test.index, columns=yr_dum_train.columns)
        X_train_ols = np.hstack([X_train_raw, yr_dum_train.values.astype(float)])
        X_test_ols = np.hstack([X_test_raw, yr_dum_test.values.astype(float)])
        ols = LinearRegression().fit(X_train_ols, y_train)
        ols_pred = ols.predict(X_test_ols)
        r2_ols = oos_r2(y_test, ols_pred, train_mean)
        rmse_ols = np.sqrt(mean_squared_error(y_test, ols_pred))
        results.append({"model": "OLS", "test_year": test_year, "oos_r2": r2_ols,
                         "rmse": rmse_ols, "n_test": len(y_test)})

        # --- LASSO ---
        scaler = StandardScaler().fit(X_train_raw)
        X_train_s, X_test_s = scaler.transform(X_train_raw), scaler.transform(X_test_raw)
        lasso = LassoCV(cv=5, random_state=42, max_iter=20000).fit(X_train_s, y_train)
        lasso_pred = lasso.predict(X_test_s)
        r2_l = oos_r2(y_test, lasso_pred, train_mean)
        rmse_l = np.sqrt(mean_squared_error(y_test, lasso_pred))
        n_sel = int(np.sum(lasso.coef_ != 0))
        results.append({"model": "LASSO", "test_year": test_year, "oos_r2": r2_l,
                         "rmse": rmse_l, "n_test": len(y_test), "n_features_selected": n_sel})

        # --- Random Forest ---
        rf = RandomForestRegressor(n_estimators=500, max_depth=4, min_samples_leaf=8,
                                    max_features="sqrt", random_state=42, n_jobs=-1).fit(X_train_raw, y_train)
        rf_pred = rf.predict(X_test_raw)
        r2_rf = oos_r2(y_test, rf_pred, train_mean)
        rmse_rf = np.sqrt(mean_squared_error(y_test, rf_pred))
        results.append({"model": "RandomForest", "test_year": test_year, "oos_r2": r2_rf,
                         "rmse": rmse_rf, "n_test": len(y_test)})

        # --- Gradient Boosting (XGBoost stand-in) ---
        gbt = HistGradientBoostingRegressor(max_iter=200, max_depth=3, learning_rate=0.03,
                                             l2_regularization=1.0, random_state=42).fit(X_train_raw, y_train)
        gbt_pred = gbt.predict(X_test_raw)
        r2_gbt = oos_r2(y_test, gbt_pred, train_mean)
        rmse_gbt = np.sqrt(mean_squared_error(y_test, gbt_pred))
        results.append({"model": "GBT_(XGBoost_standin)", "test_year": test_year, "oos_r2": r2_gbt,
                         "rmse": rmse_gbt, "n_test": len(y_test)})

        print(f"{test_year} | OLS R2={r2_ols:.4f} | LASSO R2={r2_l:.4f} (k={n_sel}) | "
              f"RF R2={r2_rf:.4f} | GBT R2={r2_gbt:.4f}")

        pred_df = test[["ric", "fiscal_year", TARGET]].copy()
        pred_df["pred_ols"] = ols_pred
        pred_df["pred_lasso"] = lasso_pred
        pred_df["pred_rf"] = rf_pred
        pred_df["pred_gbt"] = gbt_pred
        predictions_store.append(pred_df)

        # feature importance on the final fold only
        if test_year == test_years[-1]:
            for name, model, Xte in [("lasso", lasso, X_test_s), ("rf", rf, X_test_raw), ("gbt", gbt, X_test_raw)]:
                pi = permutation_importance(model, Xte, y_test, n_repeats=20, random_state=42, n_jobs=-1)
                imp = pd.DataFrame({"feature": feature_cols, "importance_mean": pi.importances_mean,
                                     "importance_std": pi.importances_std}).sort_values(
                                     "importance_mean", ascending=False)
                last_fold_importance[name] = imp

    results_df = pd.DataFrame(results)
    results_df.to_csv("outputs/model_results.csv", index=False)
    print("\nAveraged OOS R^2 / RMSE by model:")
    print(results_df.groupby("model")[["oos_r2", "rmse"]].mean())

    pd.concat(predictions_store).to_csv("outputs/oos_predictions.csv", index=False)
    for name, imp in last_fold_importance.items():
        imp.to_csv(f"outputs/perm_importance_{name}.csv", index=False)
    print("\nSaved: outputs/model_results.csv, outputs/oos_predictions.csv, outputs/perm_importance_*.csv")

if __name__ == "__main__":
    main()

## Step 5 · Portfolio backtest (original `05_portfolio_backtest.py`)

In [ ]:
"""
05_portfolio_backtest.py

Sorts firms into quintiles each year by each model's predicted
forward return, and reports the annualised return, volatility, Sharpe
ratio, and max drawdown of a long-top-quintile strategy against an
equal-weighted benchmark of all constituents.

INPUT:  outputs/oos_predictions.csv (from script 04)
OUTPUT: outputs/portfolio_backtest.csv
"""

import pandas as pd
import numpy as np

TARGET = "fwd_total_return"
MODELS = {"pred_ols": "OLS", "pred_lasso": "LASSO", "pred_rf": "RandomForest",
          "pred_gbt": "GBT_(XGBoost_standin)"}

def stats(returns):
    ann_ret = returns.mean()
    ann_vol = returns.std()
    sharpe = ann_ret / ann_vol if ann_vol > 0 else np.nan
    cum = (1 + returns / 100).cumprod()
    max_dd = (cum / cum.cummax() - 1).min()
    return ann_ret, ann_vol, sharpe, max_dd

def main():
    preds = pd.read_csv("outputs/oos_predictions.csv")

    rows = []
    for col, label in MODELS.items():
        yearly = []
        for year, g in preds.groupby("fiscal_year"):
            g = g.copy()
            g["q"] = pd.qcut(g[col], 5, labels=False, duplicates="drop")
            top = g[g["q"] == g["q"].max()]
            yearly.append({"year": year, "ret": top[TARGET].mean()})
        tq = pd.DataFrame(yearly).set_index("year")["ret"]
        ar, av, sh, dd = stats(tq)
        rows.append({"model": label, "portfolio": "Top quintile",
                     "ann_return_pct": ar, "ann_vol_pct": av, "sharpe": sh, "max_dd": dd})

    bench = preds.groupby("fiscal_year")[TARGET].mean()
    ar, av, sh, dd = stats(bench)
    rows.append({"model": "Benchmark", "portfolio": "Equal-weighted",
                 "ann_return_pct": ar, "ann_vol_pct": av, "sharpe": sh, "max_dd": dd})

    out = pd.DataFrame(rows)
    print(out.to_string(index=False))
    out.to_csv("outputs/portfolio_backtest.csv", index=False)
    print("\nSaved: outputs/portfolio_backtest.csv")
    print("\nNOTE: with only a handful of test years, these Sharpe/vol figures are")
    print("indicative, not statistically precise -- say so in your write-up.")

if __name__ == "__main__":
    main()

## Step 6 · Pre/post-2020 robustness (original `06_robustness_subperiod.py`)
Pre-2020 test years 2018–2019 (2017 excluded — 2-year training window), post-2020 test years 2021–2024.

In [ ]:
"""
06_robustness_subperiod.py

Robustness check: does the ML-vs-OLS finding hold across a structural
break (COVID market regime), or is it an artefact of one particular
period?

Same four models, same walk-forward logic as 04_run_models.py, but
test years are split into two groups instead of pooled:
  PRE  = 2017, 2018, 2019   (last full years before the COVID shock)
  POST = 2021, 2022, 2023, 2024  (2020 itself excluded -- it's the
         shock year, neither cleanly "before" nor "after")

CAVEAT (report this honestly in your write-up): the PRE group has very
short training windows (2-4 years of history), which makes those folds
noisier than the main pooled result. Treat PRE-period numbers as
indicative of direction, not precise magnitude.

INPUT:  data/modeling_panel.csv, data/retained_esg_features.txt,
        data/sector_dummy_cols.txt
OUTPUT: outputs/subperiod_robustness.csv
"""

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

TARGET = "fwd_total_return"
PRE_YEARS = [2018, 2019]   # 2017 excluded: its training window is only
                            # 2015-2016 (2 years, ~110 obs) -- too thin
                            # to support reliable estimation and swings
                            # every model's R^2 sharply negative. This
                            # is a small-sample artefact of that one
                            # fold, not a real finding -- document this
                            # exclusion explicitly in your methodology.
POST_YEARS = [2021, 2022, 2023, 2024]

def oos_r2(y_true, y_pred, y_train_mean):
    naive = np.full_like(y_true, y_train_mean, dtype=float)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - naive) ** 2)
    return 1 - ss_res / ss_tot

def run_fold(train, test, feature_cols):
    X_train_raw = train[feature_cols].values.astype(float)
    X_test_raw = test[feature_cols].values.astype(float)
    y_train = train[TARGET].values.astype(float)
    y_test = test[TARGET].values.astype(float)
    train_mean = y_train.mean()
    out = {}

    yr_dum_train = pd.get_dummies(train["fiscal_year"], prefix="yr", drop_first=True)
    yr_dum_test = pd.DataFrame(0, index=test.index, columns=yr_dum_train.columns)
    X_train_ols = np.hstack([X_train_raw, yr_dum_train.values.astype(float)])
    X_test_ols = np.hstack([X_test_raw, yr_dum_test.values.astype(float)])
    ols = LinearRegression().fit(X_train_ols, y_train)
    out["OLS"] = oos_r2(y_test, ols.predict(X_test_ols), train_mean)

    scaler = StandardScaler().fit(X_train_raw)
    X_train_s, X_test_s = scaler.transform(X_train_raw), scaler.transform(X_test_raw)
    lasso = LassoCV(cv=min(5, len(train)//10 or 2), random_state=42, max_iter=20000).fit(X_train_s, y_train)
    out["LASSO"] = oos_r2(y_test, lasso.predict(X_test_s), train_mean)

    rf = RandomForestRegressor(n_estimators=500, max_depth=4, min_samples_leaf=8,
                                max_features="sqrt", random_state=42, n_jobs=-1).fit(X_train_raw, y_train)
    out["RandomForest"] = oos_r2(y_test, rf.predict(X_test_raw), train_mean)

    gbt = HistGradientBoostingRegressor(max_iter=200, max_depth=3, learning_rate=0.03,
                                         l2_regularization=1.0, random_state=42).fit(X_train_raw, y_train)
    out["GBT_(XGBoost_standin)"] = oos_r2(y_test, gbt.predict(X_test_raw), train_mean)

    return out

def run_period(panel, feature_cols, years, period_label):
    rows = []
    for test_year in years:
        train = panel[panel["fiscal_year"] < test_year]
        test = panel[panel["fiscal_year"] == test_year]
        if len(train) < 15 or len(test) < 5:
            print(f"  [{period_label}] skipping {test_year}: insufficient data "
                  f"(train={len(train)}, test={len(test)})")
            continue
        fold_r2 = run_fold(train, test, feature_cols)
        for model, r2 in fold_r2.items():
            rows.append({"period": period_label, "test_year": test_year, "model": model, "oos_r2": r2})
    return rows

def main():
    panel = pd.read_csv("data/modeling_panel.csv")
    with open("data/retained_esg_features.txt") as f:
        kept_esg = [l.strip() for l in f if l.strip()]
    with open("data/sector_dummy_cols.txt") as f:
        sector_dummy_cols = [l.strip() for l in f if l.strip()]
    feature_cols = kept_esg + ["log_mktcap"] + sector_dummy_cols

    print("=== PRE-2020 period ===")
    pre_rows = run_period(panel, feature_cols, PRE_YEARS, "PRE-2020")
    print("\n=== POST-2020 period ===")
    post_rows = run_period(panel, feature_cols, POST_YEARS, "POST-2020")

    all_rows = pd.DataFrame(pre_rows + post_rows)
    all_rows.to_csv("outputs/subperiod_robustness_folds.csv", index=False)

    summary = all_rows.groupby(["period", "model"])["oos_r2"].mean().unstack("period")
    summary = summary[["PRE-2020", "POST-2020"]]
    print("\n=== Mean OOS R^2 by model and sub-period ===")
    print(summary.round(4))
    summary.to_csv("outputs/subperiod_robustness.csv")
    print("\nSaved: outputs/subperiod_robustness_folds.csv, outputs/subperiod_robustness.csv")

if __name__ == "__main__":
    main()

## Step 7 · Significance tests — scikit-learn stand-in version (original `07_significance_tests.py`)
This is the earlier version, run before the real XGBoost predictions existed. The dissertation reports the **real-XGBoost** version (Step 10c); this cell is kept because it is part of the original pipeline.

In [ ]:
"""
07_significance_tests.py

Tests whether the OOS R^2 differences between models (e.g. Random
Forest vs. OLS) are statistically meaningful, not just numerically
different. Point estimates alone (as in outputs/model_results.csv)
don't tell you whether a 0.03 vs. -0.07 R^2 gap could plausibly be
noise -- this script answers that.

TWO COMPLEMENTARY TESTS ARE RUN, because with only 6 walk-forward test
years, no single test is fully satisfying on its own -- reporting both
and being transparent about the small-T caveat is itself good practice
for your Methodology chapter:

1. DIEBOLD-MARIANO TEST (year-level)
   Standard approach for comparing forecast accuracy between two
   models on the same test data. For each test year, compute the mean
   loss differential (squared error of model A minus squared error of
   model B) across all firms tested that year -- giving one number per
   year (6 numbers total, matching your 6 walk-forward folds). Test
   whether the mean of these 6 year-level differentials is
   significantly different from zero via a t-test.
   CAVEAT: with only 6 data points, this test has limited power --
   report it, but don't over-interpret a non-significant result as
   proof of "no difference".

2. YEAR-BLOCK BOOTSTRAP
   Resample the 6 test years with replacement (as whole blocks, not
   individual observations, to preserve within-year correlation
   structure) 5,000 times, recompute the mean OOS R^2 for each model
   in each resample, and build a 95% confidence interval for the
   difference. More robust to the small-T problem than the DM test's
   normal-approximation assumption, and gives a confidence interval
   rather than just a p-value -- more informative for your Results
   chapter's discussion of precision.

INPUT:  outputs/oos_predictions.csv (main run) or
        outputs/subperiod_robustness_folds.csv (sub-period runs)
OUTPUT: outputs/significance_tests.csv, printed summary
"""

import pandas as pd
import numpy as np
from scipy import stats

TARGET = "fwd_total_return"
MODEL_PAIRS = [
    ("pred_rf", "pred_ols", "RandomForest", "OLS"),
    ("pred_gbt", "pred_ols", "GBT", "OLS"),
    ("pred_lasso", "pred_ols", "LASSO", "OLS"),
    ("pred_gbt", "pred_rf", "GBT", "RandomForest"),
]
N_BOOTSTRAP = 5000
RNG = np.random.default_rng(42)

def diebold_mariano_year_level(preds, col_a, col_b):
    """Returns (dm_stat, p_value, year_level_diffs) comparing squared
    errors of model A vs model B, aggregated to one value per test year."""
    preds = preds.copy()
    preds["se_a"] = (preds[TARGET] - preds[col_a]) ** 2
    preds["se_b"] = (preds[TARGET] - preds[col_b]) ** 2
    preds["loss_diff"] = preds["se_a"] - preds["se_b"]  # negative = A better

    year_diffs = preds.groupby("fiscal_year")["loss_diff"].mean()
    n = len(year_diffs)
    mean_d = year_diffs.mean()
    se_d = year_diffs.std(ddof=1) / np.sqrt(n)
    if se_d == 0:
        return np.nan, np.nan, year_diffs
    dm_stat = mean_d / se_d
    p_value = 2 * (1 - stats.t.cdf(abs(dm_stat), df=n - 1))
    return dm_stat, p_value, year_diffs

def year_block_bootstrap_r2_diff(preds, col_a, col_b):
    """Block-bootstraps whole test years to build a CI for the
    difference in OOS R^2 (using the historical-mean-of-that-year's
    training data as the naive benchmark, approximated here with the
    within-year target mean for simplicity in the bootstrap)."""
    years = preds["fiscal_year"].unique()

    def r2_for_sample(sample_df, col):
        y_true = sample_df[TARGET].values
        y_pred = sample_df[col].values
        naive = np.full_like(y_true, y_true.mean())
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - naive) ** 2)
        return 1 - ss_res / ss_tot if ss_tot != 0 else np.nan

    diffs = []
    for _ in range(N_BOOTSTRAP):
        sampled_years = RNG.choice(years, size=len(years), replace=True)
        sample_df = pd.concat([preds[preds["fiscal_year"] == y] for y in sampled_years])
        r2_a = r2_for_sample(sample_df, col_a)
        r2_b = r2_for_sample(sample_df, col_b)
        diffs.append(r2_a - r2_b)

    diffs = np.array(diffs)
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
    p_value = 2 * min((diffs > 0).mean(), (diffs < 0).mean())
    return diffs.mean(), ci_low, ci_high, p_value

def main():
    preds = pd.read_csv("outputs/oos_predictions.csv")
    print(f"Loaded {len(preds)} predictions across {preds['fiscal_year'].nunique()} test years\n")

    results = []
    for col_a, col_b, name_a, name_b in MODEL_PAIRS:
        print(f"=== {name_a} vs {name_b} ===")

        dm_stat, dm_p, year_diffs = diebold_mariano_year_level(preds, col_a, col_b)
        print(f"Diebold-Mariano (year-level, n={len(year_diffs)}): "
              f"stat={dm_stat:.3f}, p={dm_p:.3f}")

        boot_mean, ci_low, ci_high, boot_p = year_block_bootstrap_r2_diff(preds, col_a, col_b)
        print(f"Year-block bootstrap R^2 diff: mean={boot_mean:.4f}, "
              f"95% CI=[{ci_low:.4f}, {ci_high:.4f}], p={boot_p:.3f}")

        sig_note = "significant at 10%" if min(dm_p, boot_p) < 0.10 else "not significant at conventional levels"
        print(f"-> {sig_note}\n")

        results.append({
            "comparison": f"{name_a} vs {name_b}",
            "dm_stat": dm_stat, "dm_pvalue": dm_p,
            "bootstrap_r2_diff_mean": boot_mean,
            "bootstrap_ci_low": ci_low, "bootstrap_ci_high": ci_high,
            "bootstrap_pvalue": boot_p,
        })

    out = pd.DataFrame(results)
    out.to_csv("outputs/significance_tests.csv", index=False)
    print("Saved: outputs/significance_tests.csv")
    print("\nHONEST CAVEAT for your Methodology chapter: with only 6 walk-forward")
    print("test years, both tests have limited statistical power. A non-significant")
    print("p-value here does not prove models perform identically -- it means the")
    print("sample is too small to distinguish them with confidence. Report this")
    print("limitation explicitly rather than letting a p>0.10 read as a null result.")

if __name__ == "__main__":
    main()

## Step 8 · Descriptive statistics and correlations (Tables 3.2, 3.3, 3.4) — exact inline code

In [ ]:
import pandas as pd
import numpy as np

panel = pd.read_csv('data/modeling_panel.csv')
with open('data/retained_esg_features.txt') as f:
    kept_esg = [l.strip() for l in f if l.strip()]

TARGET = 'fwd_total_return'
vars_for_table = [TARGET] + kept_esg + ['log_mktcap']

desc = panel[vars_for_table].describe(percentiles=[0.5]).T
desc = desc[['count','mean','std','min','50%','max']]
desc.columns = ['N','Mean','SD','Min','Median','Max']
pd.set_option('display.width', 150)
pd.set_option('display.max_columns', 10)
print(desc.round(3))

print("\n\nSample composition:")
print("Total obs:", len(panel))
print("Unique companies:", panel['ric'].nunique())
print("Fiscal years:", sorted(panel['fiscal_year'].unique()))
print("\nSector breakdown:")
print(panel['sector'].value_counts())

In [ ]:
import pandas as pd
panel = pd.read_csv('data/modeling_panel.csv')
cols = ['fwd_total_return','ESG Score','Environmental Pillar Score','Social Pillar Score','Governance Pillar Score','log_mktcap']
corr = panel[cols].corr(method='pearson')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 10)
print(corr.round(3))
print()
print('Shareholder Rights Policy unique values:', panel['Shareholder Rights Policy'].unique())

## Step 9 · Real XGBoost + SHAP (original Colab script `04b_xgboost_and_shap_PRODUCTION.py`)
This is the script that was run on Google Colab. It needs `xgboost` and `shap` (installed in Step 0). It reads three files from the **current folder**, so 9a first copies them there.

* **9a** runs it fresh. Takes about a minute. May differ slightly from the dissertation if your `xgboost` version differs from the one Colab used.
* **9b** restores your original Colab outputs from `saved_colab_outputs/` (only copies a file if it is not already present). Run 9b if you skip 9a, or if you want the dissertation numbers exactly.

**Order matters:** if you run 9a and want the original Colab figures instead, delete the files it wrote (`xgboost_real_*.csv`, `shap_*`) and run 9b.

In [ ]:
# [[XGB_FRESH_RUN]]  (9a) -- copy the three input files next to the notebook, then run the original script
import shutil
for fn in ["modeling_panel.csv", "retained_esg_features.txt", "sector_dummy_cols.txt"]:
    shutil.copy(f"data/{fn}", fn)
print("Inputs copied. Running the original Colab script ...\n")

In [ ]:
# [[XGB_FRESH_RUN]]  (9a, original script text)
"""
04b_xgboost_and_shap_PRODUCTION.py

Run this on Google Colab or any machine with internet access.
Requires: pip install xgboost shap

Does two things your sandbox version couldn't (no internet there):
  1. Runs the SAME walk-forward evaluation as 04_run_models.py, but
     with the REAL xgboost.XGBRegressor (with early stopping) instead
     of the HistGradientBoostingRegressor stand-in.
  2. Computes REAL SHAP values (shap.TreeExplainer) on the final test
     fold for both XGBoost and Random Forest, saves a summary plot and
     a ranked mean-|SHAP| importance table.

BEFORE RUNNING: upload these three files (from your pipeline's data/
folder) to the same location as this script:
    modeling_panel.csv
    retained_esg_features.txt
    sector_dummy_cols.txt

OUTPUT (all saved to the current directory):
    xgboost_real_results.csv        -- OOS R^2/RMSE per fold, real XGBoost
    shap_importance_xgb.csv         -- ranked mean|SHAP| for XGBoost
    shap_importance_rf.csv          -- ranked mean|SHAP| for Random Forest
    shap_summary_xgb.png            -- SHAP summary plot, XGBoost
    shap_summary_rf.png             -- SHAP summary plot, Random Forest
"""

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
import shap
import matplotlib.pyplot as plt

TARGET = "fwd_total_return"
TEST_YEAR_MIN = 2019
TEST_YEAR_MAX = 2024

def oos_r2(y_true, y_pred, y_train_mean):
    naive = np.full_like(y_true, y_train_mean, dtype=float)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - naive) ** 2)
    return 1 - ss_res / ss_tot

def main():
    panel = pd.read_csv("modeling_panel.csv")
    with open("retained_esg_features.txt") as f:
        kept_esg = [l.strip() for l in f if l.strip()]
    with open("sector_dummy_cols.txt") as f:
        sector_dummy_cols = [l.strip() for l in f if l.strip()]
    feature_cols = kept_esg + ["log_mktcap"] + sector_dummy_cols

    all_years = sorted(panel["fiscal_year"].unique())
    test_years = [y for y in all_years if TEST_YEAR_MIN <= y <= TEST_YEAR_MAX]
    print("Test years:", test_years)

    results = []
    predictions_store = []
    last_fold_data = None  # keep the final fold's train/test for SHAP

    for test_year in test_years:
        train = panel[panel["fiscal_year"] < test_year]
        test = panel[panel["fiscal_year"] == test_year]
        if len(train) < 30 or len(test) < 5:
            continue

        X_train_full = train[feature_cols].values.astype(float)
        X_test = test[feature_cols].values.astype(float)
        y_train_full = train[TARGET].values.astype(float)
        y_test = test[TARGET].values.astype(float)
        train_mean = y_train_full.mean()

        # hold out 15% of training data as an early-stopping validation set
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full, test_size=0.15, random_state=42
        )

        model = XGBRegressor(
            n_estimators=1000, max_depth=4, learning_rate=0.03,
            subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=1.0,
            early_stopping_rounds=50, eval_metric="rmse", random_state=42
        )
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

        preds = model.predict(X_test)
        r2 = oos_r2(y_test, preds, train_mean)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        results.append({"test_year": test_year, "oos_r2": r2, "rmse": rmse,
                         "best_iteration": model.best_iteration})
        print(f"{test_year} | XGBoost (real) OOS R^2={r2:.4f}  RMSE={rmse:.4f}  "
              f"best_iter={model.best_iteration}")

        # save individual predictions for significance testing later
        pred_df = test[["ric", "fiscal_year", TARGET]].copy()
        pred_df["pred_xgb_real"] = preds
        predictions_store.append(pred_df)

        if test_year == test_years[-1]:
            last_fold_data = (X_train_full, X_test, y_train_full, y_test, model)

    results_df = pd.DataFrame(results)
    results_df.to_csv("xgboost_real_results.csv", index=False)
    print(f"\nMean OOS R^2 (real XGBoost): {results_df['oos_r2'].mean():.4f}")
    print("Saved: xgboost_real_results.csv")

    pd.concat(predictions_store).to_csv("xgboost_real_predictions.csv", index=False)
    print("Saved: xgboost_real_predictions.csv (for significance testing)")

    # --- Real SHAP analysis on the final fold ---
    X_train_full, X_test, y_train_full, y_test, xgb_model = last_fold_data

    print("\nComputing SHAP values for XGBoost...")
    explainer_xgb = shap.TreeExplainer(xgb_model)
    shap_values_xgb = explainer_xgb.shap_values(X_test)

    shap.summary_plot(shap_values_xgb, X_test, feature_names=feature_cols, show=False)
    plt.tight_layout()
    plt.savefig("shap_summary_xgb.png", bbox_inches="tight", dpi=150)
    plt.close()

    imp_xgb = pd.DataFrame({
        "feature": feature_cols,
        "mean_abs_shap": np.abs(shap_values_xgb).mean(axis=0)
    }).sort_values("mean_abs_shap", ascending=False)
    imp_xgb.to_csv("shap_importance_xgb.csv", index=False)
    print("Top 10 features by SHAP (XGBoost):")
    print(imp_xgb.head(10).to_string(index=False))

    # Also do Random Forest for comparison (same final fold)
    print("\nFitting Random Forest on the same final fold for SHAP comparison...")
    rf = RandomForestRegressor(n_estimators=500, max_depth=4, min_samples_leaf=8,
                                max_features="sqrt", random_state=42, n_jobs=-1)
    rf.fit(X_train_full, y_train_full)
    explainer_rf = shap.TreeExplainer(rf)
    shap_values_rf = explainer_rf.shap_values(X_test)

    shap.summary_plot(shap_values_rf, X_test, feature_names=feature_cols, show=False)
    plt.tight_layout()
    plt.savefig("shap_summary_rf.png", bbox_inches="tight", dpi=150)
    plt.close()

    imp_rf = pd.DataFrame({
        "feature": feature_cols,
        "mean_abs_shap": np.abs(shap_values_rf).mean(axis=0)
    }).sort_values("mean_abs_shap", ascending=False)
    imp_rf.to_csv("shap_importance_rf.csv", index=False)
    print("Top 10 features by SHAP (Random Forest):")
    print(imp_rf.head(10).to_string(index=False))

    print("\nSaved: shap_summary_xgb.png, shap_summary_rf.png, "
          "shap_importance_xgb.csv, shap_importance_rf.csv")
    print("\nDownload all output files from Colab's file browser (left sidebar)")
    print("before you close the session -- Colab deletes files when it disconnects.")

if __name__ == "__main__":
    main()

In [ ]:
# (9b) Restore the original Colab outputs if they are not already in this folder
import os, shutil
colab_files = ["xgboost_real_predictions.csv", "xgboost_real_results.csv",
               "shap_importance_xgb.csv", "shap_importance_rf.csv",
               "shap_summary_xgb.png", "shap_summary_rf.png"]
for fn in colab_files:
    if os.path.exists(fn):
        print("already present:", fn)
    elif os.path.exists(f"saved_colab_outputs/{fn}"):
        shutil.copy(f"saved_colab_outputs/{fn}", fn)
        print("restored from saved_colab_outputs/:", fn)
    else:
        print("MISSING:", fn)

## Step 10 · Real-XGBoost significance tests and backtest (Tables 5.3, 5.7, 5.8) — exact inline code
**10a** finds the duplicate firm-year rows (the 34 rows described in Chapter 3.2). **10b** merges the real XGBoost predictions with the other models' predictions. **10c** is the significance test reported in Table 5.3 and Table 5.7. **10d** is the real-XGBoost portfolio backtest in Table 5.8.

In [ ]:
# (10a) duplicate check -- why de-duplication is needed before merging (path change only: local file)
import pandas as pd
existing = pd.read_csv('outputs/oos_predictions.csv')
xgb_real = pd.read_csv('xgboost_real_predictions.csv')

print('Existing dupes on ric+fiscal_year:', existing.duplicated(subset=['ric','fiscal_year']).sum())
print('XGB real dupes on ric+fiscal_year:', xgb_real.duplicated(subset=['ric','fiscal_year']).sum())

dupe_rics = existing[existing.duplicated(subset=['ric','fiscal_year'], keep=False)]
print(dupe_rics.sort_values(['ric','fiscal_year']).head(10))

In [ ]:
# (10b) de-duplicate and merge real XGBoost predictions (path change only: local file)
import pandas as pd
existing = pd.read_csv('outputs/oos_predictions.csv')
xgb_real = pd.read_csv('xgboost_real_predictions.csv')

existing_dedup = existing.drop_duplicates(subset=['ric','fiscal_year'], keep='first')
xgb_dedup = xgb_real.drop_duplicates(subset=['ric','fiscal_year'], keep='first')

print('Existing after dedup:', existing_dedup.shape)
print('XGB real after dedup:', xgb_dedup.shape)

merged = existing_dedup.merge(xgb_dedup[['ric','fiscal_year','pred_xgb_real']], on=['ric','fiscal_year'], how='inner')
print('Merged shape:', merged.shape)
merged.to_csv('outputs/oos_predictions_with_real_xgb.csv', index=False)

In [ ]:
# (10c) significance tests with real XGBoost  ->  Table 5.3 and Table 5.7
import pandas as pd
import numpy as np
from scipy import stats

TARGET = "fwd_total_return"
N_BOOTSTRAP = 5000
RNG = np.random.default_rng(42)

preds = pd.read_csv('outputs/oos_predictions_with_real_xgb.csv')

MODEL_PAIRS = [
    ("pred_xgb_real", "pred_ols", "XGBoost(real)", "OLS"),
    ("pred_rf", "pred_ols", "RandomForest", "OLS"),
    ("pred_lasso", "pred_ols", "LASSO", "OLS"),
    ("pred_xgb_real", "pred_rf", "XGBoost(real)", "RandomForest"),
    ("pred_xgb_real", "pred_gbt", "XGBoost(real)", "GBT(sklearn standin)"),
]

def diebold_mariano_year_level(preds, col_a, col_b):
    preds = preds.copy()
    preds["se_a"] = (preds[TARGET] - preds[col_a]) ** 2
    preds["se_b"] = (preds[TARGET] - preds[col_b]) ** 2
    preds["loss_diff"] = preds["se_a"] - preds["se_b"]
    year_diffs = preds.groupby("fiscal_year")["loss_diff"].mean()
    n = len(year_diffs)
    mean_d = year_diffs.mean()
    se_d = year_diffs.std(ddof=1) / np.sqrt(n)
    if se_d == 0:
        return np.nan, np.nan
    dm_stat = mean_d / se_d
    p_value = 2 * (1 - stats.t.cdf(abs(dm_stat), df=n - 1))
    return dm_stat, p_value

def year_block_bootstrap_r2_diff(preds, col_a, col_b):
    years = preds["fiscal_year"].unique()
    def r2_for_sample(sample_df, col):
        y_true = sample_df[TARGET].values
        y_pred = sample_df[col].values
        naive = np.full_like(y_true, y_true.mean())
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - naive) ** 2)
        return 1 - ss_res / ss_tot if ss_tot != 0 else np.nan
    diffs = []
    for _ in range(N_BOOTSTRAP):
        sampled_years = RNG.choice(years, size=len(years), replace=True)
        sample_df = pd.concat([preds[preds["fiscal_year"] == y] for y in sampled_years])
        diffs.append(r2_for_sample(sample_df, col_a) - r2_for_sample(sample_df, col_b))
    diffs = np.array(diffs)
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
    p_value = 2 * min((diffs > 0).mean(), (diffs < 0).mean())
    return diffs.mean(), ci_low, ci_high, p_value

results = []
for col_a, col_b, name_a, name_b in MODEL_PAIRS:
    dm_stat, dm_p = diebold_mariano_year_level(preds, col_a, col_b)
    boot_mean, ci_low, ci_high, boot_p = year_block_bootstrap_r2_diff(preds, col_a, col_b)
    print(f"{name_a} vs {name_b}: DM p={dm_p:.4f} | Bootstrap mean_diff={boot_mean:.4f}, CI=[{ci_low:.4f},{ci_high:.4f}], p={boot_p:.4f}")
    results.append({"comparison": f"{name_a} vs {name_b}", "dm_stat": dm_stat, "dm_pvalue": dm_p,
                     "bootstrap_r2_diff_mean": boot_mean, "bootstrap_ci_low": ci_low,
                     "bootstrap_ci_high": ci_high, "bootstrap_pvalue": boot_p})

pd.DataFrame(results).to_csv('outputs/significance_tests_real_xgb.csv', index=False)
print("\nSaved: outputs/significance_tests_real_xgb.csv")

In [ ]:
# (10d) portfolio backtest with real XGBoost predictions  ->  Table 5.8 (XGBoost row)
import pandas as pd
import numpy as np

preds = pd.read_csv('outputs/oos_predictions_with_real_xgb.csv')
TARGET = 'fwd_total_return'

def stats(returns):
    ann_ret = returns.mean()
    ann_vol = returns.std()
    sharpe = ann_ret/ann_vol if ann_vol>0 else np.nan
    cum = (1+returns/100).cumprod()
    dd = (cum/cum.cummax()-1).min()
    return ann_ret, ann_vol, sharpe, dd

yearly = []
for year, g in preds.groupby('fiscal_year'):
    g = g.copy()
    g['q'] = pd.qcut(g['pred_xgb_real'], 5, labels=False, duplicates='drop')
    top = g[g['q']==g['q'].max()]
    yearly.append({'year':year,'ret':top[TARGET].mean()})
tq = pd.DataFrame(yearly).set_index('year')['ret']
ar,av,sh,dd = stats(tq)
print(f'XGBoost (real) top-quintile: ann_return={ar:.3f} vol={av:.3f} sharpe={sh:.3f} maxdd={dd:.3f}')

# --- (added) save the row so the recap cell can include it in the Table 5.8 view ---
pd.DataFrame([{'model':'XGBoost (real)','ann_return_pct':ar,'ann_vol_pct':av,'sharpe':sh,'max_dd':dd}]).to_csv(
    'outputs/portfolio_backtest_real_xgb.csv', index=False)

## Step 11 · Recap tables and self-check **(added — convenience cell, not part of the original pipeline)**
Prints the dissertation's headline tables from the files produced above, then compares every output file against `expected_outputs/` (the results from the original run) and reports the largest absolute difference. **`0.00e+00` everywhere means this notebook reproduced your dissertation numbers exactly.**

In [ ]:
import pandas as pd, numpy as np, os, glob
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 20)

mr = pd.read_csv('outputs/model_results.csv')
xg = pd.read_csv('xgboost_real_results.csv')

# ---- Table 5.1 : mean OOS R^2 and RMSE
t51 = mr[mr['model'].isin(['OLS','LASSO','RandomForest'])].groupby('model')[['oos_r2','rmse']].mean().reindex(['OLS','LASSO','RandomForest'])
t51.loc['XGBoost (real)'] = [xg['oos_r2'].mean(), xg['rmse'].mean()]
print("TABLE 5.1  Mean out-of-sample R^2 and RMSE (2019-2024)"); print(t51.round(3)); print()

# ---- Table 5.2 : by test year
t52 = mr[mr['model'].isin(['OLS','LASSO','RandomForest'])].pivot(index='test_year', columns='model', values='oos_r2')[['OLS','LASSO','RandomForest']]
t52['XGBoost (real)'] = xg.set_index('test_year')['oos_r2']
print("TABLE 5.2  OOS R^2 by test year"); print(t52.round(3)); print()

# ---- Table 5.3 / 5.7 : significance tests with real XGBoost
print("TABLE 5.3 / 5.7  Significance tests"); print(pd.read_csv('outputs/significance_tests_real_xgb.csv').round(4)); print()

# ---- Table 5.4 : SHAP importance, XGBoost
print("TABLE 5.4  Top-10 SHAP importance, XGBoost"); print(pd.read_csv('shap_importance_xgb.csv').head(10).round(3)); print()

# ---- Table 5.6 : sub-period robustness
print("TABLE 5.6  Mean OOS R^2 by sub-period (XGBoost row here is the sklearn stand-in)"); print(pd.read_csv('outputs/subperiod_robustness.csv').round(3)); print()

# ---- Table 5.8 : portfolio backtest
pb = pd.read_csv('outputs/portfolio_backtest.csv')[['model','ann_return_pct','ann_vol_pct','sharpe','max_dd']]
pb = pb[pb['model']!='GBT_(XGBoost_standin)']
pb = pd.concat([pb, pd.read_csv('outputs/portfolio_backtest_real_xgb.csv')], ignore_index=True)
print("TABLE 5.8  Top-quintile portfolio backtest (6 annual observations)"); print(pb.round(2)); print()

# ---- Self-check against the original run
print("SELF-CHECK vs expected_outputs/  (max absolute difference per file; 0.00e+00 = exact match)")
checks = {}
for exp in sorted(glob.glob('expected_outputs/*.csv')):
    name = os.path.basename(exp)
    for cand in [f'outputs/{name}', name, f'data/{name}']:
        if os.path.exists(cand):
            a, b = pd.read_csv(cand), pd.read_csv(exp)
            if a.shape != b.shape:
                checks[name] = f"SHAPE DIFFERS {a.shape} vs {b.shape}"
            else:
                num = a.select_dtypes('number').columns
                checks[name] = f"{float(np.nanmax(np.abs(a[num].values - b[num].values))):.2e}" if len(num) else "no numeric cols"
            break
    else:
        checks[name] = "not produced (skipped step?)"
for k, v in checks.items():
    print(f"  {k:48s} {v}")